# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Display high-level metadata information
print(f"Dataset name: {metadata.name}\n")
print(f"Dataset version: {metadata.version}")
print(f"Authors:")
for idx, author in enumerate(getattr(metadata, 'author', [])):
    print(f"  {idx+1}. {getattr(author, '@id', str(author))}")

print("\nTop-level record sets (by @id):")
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"  - {rs['@id']}")
elif hasattr(metadata, 'recordSet'):
    for rs in metadata.recordSet:
        print(f"  - {rs['@id']}")
else:
    print("  [No record sets found directly in metadata. Discovering via dataset.record_sets()...")

# Use dataset API to discover all record sets and their fields by @id
record_sets = list(dataset.record_sets())
print(f"\nAll discovered record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    # List their fields
    fields = rs.get('fields', [])
    if not fields and 'field' in rs:
        fields = rs['field']
    if fields:
        print("    Fields:")
        for f in fields:
            # Some fields are dict, some are just @id
            if isinstance(f, dict):
                print(f"      - {f.get('@id')}: {f.get('name')}")
            else:
                print(f"      - {f}")
    if 'columns' in rs:
        print("    Columns:")
        for c in rs['columns']:
            print(f"      - {c['@id']}: {c.get('name')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Discover all available record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
print("Discovered record sets:")
for rsid in record_set_ids:
    print(f"  {rsid}")

# For this dataset, with one main record set, let's extract all and show main one
dataframes = {}
for rsid in record_set_ids:
    print(f"\nLoading records from record set: {rsid}")
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# For demo, show from the first record set
main_record_set_id = record_set_ids[0]
print(f"\nAvailable columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping by an attribute, using `@id` references for each field.

In [ ]:
# Suppose based on the schema you identified a numeric field 'phq9_total_score' (PHQ-9 depression score)
# For demonstration, reference by @id as required (adjust as needed if your schema uses another @id)
# Replace with the correct @id from the metadata overview above if different
numeric_field = 'phq9_total_score'  # Example field name: PHQ-9 total score
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Check if the field exists, convert if needed
if numeric_field in df.columns:
    # Remove missing values for the field
    df_nona = df.dropna(subset=[numeric_field])
    # For demonstration, filter for likely non-zero, non-null scores
    threshold = 10
    filtered_df = df_nona[df_nona[numeric_field] > threshold].copy()
    print(f"Filtered records with '{@id}={numeric_field}' > {threshold} (count: {len(filtered_df)}):")
    print(filtered_df[[numeric_field]].head())
    # Normalize (z-score)
    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())
    # Group by a demographic field, e.g., 'gender' if exists, else skip
    group_field = 'gender'  # Replace with actual demographic @id as per the schema
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[[numeric_field, normalized_col]].mean()
        print(f"\nMean {numeric_field} by {group_field} for filtered subset:")
        print(grouped_df)
else:
    print(f"Field {numeric_field} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of PHQ-9 scores
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field} (PHQ-9 Scores)')
    plt.xlabel('PHQ-9 Total Score')
    plt.ylabel('Number of Participants')
    plt.show()
    
    # Boxplot by gender if available
    if group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel('PHQ-9 Total Score')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to load a Croissant-specified dataset using the `mlcroissant` library and explored the FAIR² Mental Health Survey data from Kilifi County, Kenya by referencing all record sets and fields via their respective `@id` values. We conducted basic exploratory data analysis on the depression scores (PHQ-9), including filtering, normalization, and grouping by demographics, and visualized field distributions. Further analysis could include exploring other mental health measures, more granular demographic analysis, and longitudinal trends if time-series data are available.